# 07_03 A recurrent network: can a network read a sentence in order?

Everything so far has thrown word order away: a bag of words, an average of embeddings. A **recurrent
neural network** (RNN) reads one word at a time and carries a hidden state forward, so in principle it can
use order. This notebook builds the book's Chapter 7 classifier, finds the bug in it, builds it properly in
PyTorch, and then measures, rather than assumes, how much of the sentence it actually remembers.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-07-how-a-network-learns", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'nltk': 'nltk',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'matplotlib': 'matplotlib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    import nltk
    for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng', 'maxent_ne_chunker_tab', 'words', 'reuters']:
        nltk.download(pkg, quiet=True)
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import time
import torch
import newswire
from nlpcheck import ask, guess, reveal, check_07_03

torch.set_num_threads(4)
data = newswire.load_split()
X_train, y_train, X_test, y_test = data["X_train"], data["y_train"], data["X_test"], data["y_test"]
results = {"majority": {"accuracy": round(float((y_test == 0).float().mean()), 4), "macro_f1": None}}
print("always guessing 'earn' scores", results["majority"]["accuracy"], "accuracy on the test split")

## 1. Recall

**r5.** Momentum 0.9 on a steady slope makes the effective step about how many times the plain step?
(a) 0.9, (b) 1.9, (c) 10

**r6.** What does a learning rate far too large do? (a) the loss grows until it is nan, (b) training gets
faster, (c) nothing, the optimiser corrects it

In [ ]:
ask("r5", "")
ask("r6", "")

## 2. How a newswire becomes a sequence

Each newswire is its first 50 words, each replaced by its rank in the training vocabulary (1 means a word
too rare to be in the 10,000; 0 is padding after the last word). The first eight ids of the first test
newswire, and the words behind them:

In [ ]:
words = {i: w for w, i in data["vocab"].items()}
ids = X_test[0][:8].tolist()
print(ids)
print([words.get(i, "<rare>" if i == 1 else "<pad>") for i in ids])

## 3. The book's network, exactly as it was built

The book's notebook reshaped these ids into shape `(documents, 50, 1)` and fed them straight to a
`SimpleRNN`: at each step, one input number, the word's id. `newswire.ScalarIdRNN` rebuilds that in PyTorch.
Predict its test accuracy, knowing that always guessing "earn" already scores 0.505. Training takes up to
about half a minute.

In [ ]:
guess("book_accuracy", None)   # a number between 0 and 1

In [ ]:
torch.manual_seed(0)
t = time.time()
book = newswire.ScalarIdRNN()
newswire.train(book, X_train, y_train, epochs=5)
results["scalar_ids"] = newswire.evaluate(book, X_test, y_test)
print(results["scalar_ids"], f"({time.time() - t:.0f} s)")
reveal("book_accuracy", round(results["scalar_ids"]["accuracy"], 2))

About 0.60, a little above always guessing, and a macro-F1 near 0.2: it barely learned. The mechanism is in
what the network was told. An id is a **name**, not a **quantity**: word 1,523 is not "more" than word 12,
and a word with id 40 is no more like one with id 41 than like one with id 9,000. Fed in as one number, the
ids told the network that "oil" is roughly "profit" plus a bit. The fix, in every language model since, is an
**embedding layer**: each id looks up its own learned vector, so similarity is something the network learns
rather than an accident of numbering.

## 4. The same RNN, with embeddings

`newswire.RNNClassifier` embeds each word as 64 numbers, runs `nn.RNN` left to right, and classifies from the
hidden state after the last real word. Training takes between half a minute and a minute.

In [ ]:
torch.manual_seed(0)
t = time.time()
rnn = newswire.RNNClassifier()
newswire.train(rnn, X_train, y_train, epochs=6)
results["rnn"] = newswire.evaluate(rnn, X_test, y_test)
print(results["rnn"], f"({time.time() - t:.0f} s)")

Much better: about 0.81 accuracy. But the macro-F1, the average over the six topics, is only about 0.52,
which means the small topics (crude, trade, interest) are mostly missed. Now the question this lab has been
heading for. Lab 02's lesson was that throwing order away costs meaning. So an RNN, which reads in order,
should beat a model that simply averages the same embeddings, with no order at all. Predict: will it?

In [ ]:
guess("rnn_beats_bag", None)   # "yes" or "no" 

In [ ]:
torch.manual_seed(0)
t = time.time()
bag = newswire.BagClassifier()
newswire.train(bag, X_train, y_train, epochs=5)
results["bag"] = newswire.evaluate(bag, X_test, y_test)
print("bag of embeddings:", results["bag"], f"({time.time() - t:.0f} s)")
print("RNN:              ", results["rnn"])
reveal("rnn_beats_bag", "yes" if results["rnn"]["macro_f1"] > results["bag"]["macro_f1"] else "no")

No: the average of the embeddings reaches about 0.93 accuracy and 0.81 macro-F1, far ahead. The RNN reads in
order, and the order is not what it lost. Its prediction comes only from the hidden state after the last
word, so everything it read has to survive the whole journey to the end. The next section measures how much
survives.

## 5. How far back does the error reach?

To learn from a word, the network needs that word's gradient: how much the loss would change if that word's
embedding changed. The cell below asks for the gradient at every one of the 50 positions, for full-length
newswires, and compares the first word's with the last word's, first for an untrained RNN and then for the
one you just trained. Predict the untrained ratio, first over last, as a power of ten, for example `1e-3`.

In [ ]:
guess("grad_ratio", None)

In [ ]:
def position_gradients(model, n=256):
    model.eval()
    x = X_train[:n]
    emb = model.emb(x).detach().requires_grad_(True)       # the embeddings, as a leaf we can ask about
    lengths = (x != 0).sum(1).clamp(min=1)
    states, _ = model.rnn(emb)
    logits = model.out(states[torch.arange(n), lengths - 1])
    torch.nn.functional.cross_entropy(logits, y_train[:n]).backward()
    full = lengths == newswire.MAX_LEN                     # only newswires that fill all 50 positions
    return emb.grad.norm(dim=2)[full].mean(0)              # the gradient's size at each position

torch.manual_seed(0)
g_untrained = position_gradients(newswire.RNNClassifier())
g_trained = position_gradients(rnn)
for name, g in (("untrained", g_untrained), ("trained", g_trained)):
    print(f"{name:9}  last {g[-1]:.1e}   10 back {g[-11]:.1e}   25 back {g[-26]:.1e}   first {g[0]:.1e}")
results["grad_ratio_untrained"] = float(g_untrained[0] / g_untrained[-1])
results["grad_ratio_trained"] = float(g_trained[0] / g_trained[-1])
reveal("grad_ratio", f"{results['grad_ratio_untrained']:.0e}")

Around `1e-17` for the untrained network: the first word's gradient is about a hundred million billion times
smaller than the last word's. After training it is about `1e-2`, better, and still a hundredth.

The mechanism is the chain rule from the first notebook. Going back one step in time multiplies the gradient
by the recurrent weights and by the slope of `tanh`, which is at most 1 and usually well below it. Fifty steps
back, it has been multiplied by that factor fifty times. A factor of 0.45, fifty times over, is about `5e-18`,
the same size as what you just measured.
This is the **vanishing gradient** the chapter ends on, and the reason the plain RNN learned mostly from the
last few words of each lead. (If the factor were above 1, the same multiplication would **explode**; that is
why `newswire.train` clips the gradient's size at 5.)

## 6. Your turn: stop relying on the last state

One remedy needs no new kind of network: instead of classifying from the last hidden state only, average the
hidden states at every real word, so each word has a short path to the output. `newswire.RNNMeanClassifier`
does exactly that. Train it the same way as the RNN in section 4 (6 epochs, seed 0) and evaluate it. Between
half a minute and a minute.

In [ ]:
torch.manual_seed(0)
t = time.time()
rnn_mean = None                 # YOUR CODE HERE: build a newswire.RNNMeanClassifier
# YOUR CODE HERE: train it on X_train, y_train for 6 epochs with newswire.train
results["rnn_mean"] = newswire.evaluate(rnn_mean, X_test, y_test) if rnn_mean is not None else None
print(results["rnn_mean"], f"({time.time() - t:.0f} s)")

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump(results, open("out/07_03_results.json", "w"), indent=1)
check_07_03()

Pooling brings the RNN up to about the bag's level. Which says something honest about this task: newswire
topics are mostly decided by which words appear, and order adds little. Order matters where the meaning turns
on it ("not good", "rates rise after inflation falls"), and for those tasks an RNN that can actually remember
is needed. That is the **LSTM**, which the next lab builds: an RNN with a separate memory lane that the
gradient can travel back along without being multiplied down to nothing.

## 7. Exit ticket

**x3.** Why does the gradient at the first word of a long sequence vanish in a plain RNN? (a) the first word
is padding, (b) the optimiser skips it, (c) going back each step multiplies it by a factor below 1, fifty
times over

In [ ]:
ask("x3", "")